## Read and Write Data from Oracle to Postgre DB using DuckDB and Polars

##### ✅ Step 1: Read Data from Oracle

In [ ]:
import duckdb
import oracledb
import polars as pl
from datetime import datetime, date

# ─── Oracle credentials ───────────────────────────────────────────────────────
ORA_USER     = "username"
ORA_PASSWORD = "password"
ORA_HOST     = "hostname"
ORA_PORT     = 1521
ORA_SERVICE  = "service_name"

oracledb.init_oracle_client(lib_dir=r"C:\Oracle\instantclient_23_0")

QUERY = f"""
    SELECT * FROM 
"""
# ─── Extract from Oracle ──────────────────────────────────────────────────────
with oracledb.connect(
    user=ORA_USER,
    password=ORA_PASSWORD,
    host=ORA_HOST,
    port=ORA_PORT,
    service_name=ORA_SERVICE
) as conn:

    cur = conn.cursor()
    cur.execute(QUERY)

    columns = [c[0].lower() for c in cur.description]
    rows = cur.fetchall()

##### ✅ Step 2: Normalize Oracle datatypes

In [ ]:
def clean_value(v):
    if isinstance(v, (datetime, date)):
        return v.isoformat()
    return v

cleaned_rows = [tuple(clean_value(v) for v in r) for r in rows]

##### ✅ Step 3: Load Data into DuckDB (staging)

In [ ]:
# ─── Load into DuckDB ───────────────────────────────────────────────────────
duck = duckdb.connect(":memory:")

col_defs  = ", ".join([f'"{c}" VARCHAR' for c in columns])
col_names = ", ".join([f'"{c}"' for c in columns])
marks     = ", ".join(["?"] * len(columns))

duck.execute(f"CREATE TABLE tableName_raw ({col_defs})")

duck.executemany(
    f"INSERT INTO tableName_raw ({col_names}) VALUES ({marks})",
    cleaned_rows
)

##### ✅ Step 4: DuckDB → Polars (zero-copy)

In [ ]:
# ─── Convert to Polars DataFrame ───────────────────────────────
arrow_table = duck.execute(
    "SELECT * FROM tableName_raw"
).to_arrow_table()

df = pl.from_arrow(arrow_table)

print(df)

##### ✅ Step 5: Add Business & Ingestion Datae

In [ ]:
from datetime import date, timedelta
# ingestion_date is the date when the data is ingested into the lakehouse, 
# and business_date is the date when the data is generated in the source system. 
# You can adjust the logic to calculate these dates based on your specific requirements.
df = df.with_columns([
    pl.lit(date.today()).alias("ingestion_date"),
    pl.lit(date.today() - timedelta(days=1)).alias("business_date")
])

print(df.shape)

##### ✅ Step 6: Register Polars DataFrame into DuckDB

In [ ]:
duck.register("pl_df", df)

##### Create DuckDB MERGE staging table

In [ ]:
# Convert the "timestamp" column to TIMESTAMP type in DuckDB. Adjust the format string if your timestamp format is different.
duck.execute("""
    CREATE OR REPLACE TABLE tableName_stage AS
    SELECT * EXCLUDE ("timestamp"), 
        strptime("timestamp", '%Y-%m-%dT%H:%M:%S')::TIMESTAMP 
        AS "timestamp" FROM pl_df
""")

##### ✅ Step 7: Write DuckDB → Attach PostgreSQL

In [ ]:
# Replace the connection string with your actual PostgreSQL connection details.
PG_CONN = (
    "host=localhost "
    "port=5432 "
    "dbname=postgres "
    "user=postgres "
    "password=postgres"
)

duck.execute("INSTALL postgres;")
duck.execute("LOAD postgres;")

duck.execute(f"""
    ATTACH OR REPLACE '{PG_CONN}' AS pg (TYPE postgres);
""")

##### Step 8: Initial load (First Time Only)

In [ ]:
# Create the target table in PostgreSQL if it doesn't exist, and insert data from the staging table.
duck.execute(f"""
    CREATE TABLE IF NOT EXISTS pg.public.tableName
    AS SELECT * FROM tableName_stage
""")

##### Step 9: Incremental MERGE

In [ ]:
# Perform the upsert operation using MERGE statement in PostgreSQL. Adjust the column names and table names as needed.
duck.execute(f"""
    MERGE INTO pg.public.tableName AS t
    USING tableName_stage AS s
    ON
        t.col1      = s.col1 AND
        t.col2      = s.col2            # col1 and col2 are the primary keys of the table.

    WHEN MATCHED THEN
    UPDATE SET
        col3 = s.part_no_val0, 
        col4 = s.mo_parts_in_val0, 
        col5 = s.mo_parts_out_val0

    WHEN NOT MATCHED THEN
    INSERT (
        col1, col2, col3, col4, col5, ingestion_date, business_date
    )
             
    VALUES (
        s.col1, s.col2, s.col3, s.col4, s.col5, s.ingestion_date, s.business_date
    )
""");

##### Step 10: PostgreSQL Index (Mandatory) (First time only)

In [ ]:
duck.execute(f"""
    CREATE UNIQUE INDEX tableName_pk
    ON tableName_raw (
        col1,
        col2
    );
""")